# 01. Data Understanding - Customer Churn Analysis

## Executive Summary
Customer churn represents one of the most critical challenges for subscription-based telecommunication companies. Acquiring a new customer is estimated to cost 5 to 7 times more than retaining an existing one. Predicting churn allows proactive engagement with at-risk accounts.

### Objectives of this Notebook:
1. Ingest the IBM Telco Customer Churn dataset safely across platforms.
2. Inspect dataset dimensions, feature data types, and structural properties.
3. Identify missing values, blank string anomalies, and duplicate records.
4. Quantify class distribution of the target variable (`Churn`) and establish baseline churn rate.
5. Catalog numerical versus categorical attributes for subsequent processing.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Display configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version : {np.__version__}")

Pandas version: 2.3.3
NumPy version : 2.4.1


## 1. Data Ingestion
We locate the raw dataset using relative path resolution supporting both case variants (`Customer_Churn.csv` and `customer_churn.csv`).

In [2]:
candidates = [
    Path("../data/raw/Customer_Churn.csv"),
    Path("../data/raw/customer_churn.csv"),
    Path("data/raw/Customer_Churn.csv"),
    Path("data/raw/customer_churn.csv"),
]

raw_path = next((p for p in candidates if p.is_file()), None)
if raw_path is None:
    raise FileNotFoundError("Raw customer churn dataset could not be located.")

print(f"Loaded raw dataset from: {raw_path.resolve()}")
df = pd.read_csv(raw_path)

Loaded raw dataset from: D:\Projects\AIML\Customer_Churn_Prediction\data\raw\Customer_Churn.csv


## 2. Dataset Structure & Dimensions

In [3]:
print(f"Total Rows (Customers): {df.shape[0]:,}")
print(f"Total Columns (Features + Target): {df.shape[1]}")
df.head(5)

Total Rows (Customers): 7,043
Total Columns (Features + Target): 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.tail(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes
7042,3186-AJIEK,Male,0,No,No,66,Yes,No,Fiber optic,Yes,No,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.65,6844.5,No


In [5]:
df.sample(5, random_state=42)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
185,1024-GUALD,Female,0,Yes,No,1,No,No phone service,DSL,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,24.80,24.8,Yes
2715,0484-JPBRU,Male,0,No,No,41,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),25.25,996.45,No
3825,3620-EHIMZ,Female,0,Yes,Yes,52,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.35,1031.7,No
1807,6910-HADCM,Female,0,No,No,1,Yes,No,Fiber optic,No,No,Yes,No,No,No,Month-to-month,No,Electronic check,76.35,76.35,Yes
132,8587-XYZSF,Male,0,No,No,67,Yes,No,DSL,No,No,No,Yes,No,No,Two year,No,Bank transfer (automatic),50.55,3260.1,No


## 3. Schema & Data Types Inspection
Notice that `TotalCharges` is stored as an `object` (string) rather than floating point numeric. We investigate this anomaly below.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [7]:
dtype_counts = df.dtypes.value_counts()
print("Data Types Count:")
print(dtype_counts)

Data Types Count:
object     18
int64       2
float64     1
Name: count, dtype: int64


## 4. Missing Value and Whitespace Audit
Standard `.isnull().sum()` checks indicate 0 missing values. However, checking for blank spaces reveals hidden missing values in `TotalCharges`.

In [8]:
print("Standard Missing Values:")
print(df.isnull().sum())

Standard Missing Values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [9]:
blank_total_charges = (df["TotalCharges"] == " ").sum()
print(f"Whitespace-only strings in 'TotalCharges': {blank_total_charges}")

# Inspect rows where TotalCharges is blank
df_blank = df[df["TotalCharges"] == " "]
df_blank[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

Whitespace-only strings in 'TotalCharges': 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


### Key Observation on TotalCharges:
All 11 customers with blank `TotalCharges` have `tenure == 0` months and `Churn == 'No'`. These are newly onboarded customers who have not completed their first billing cycle. Setting `TotalCharges = 0.0` for these accounts is logically and domain-justified.

## 5. Duplicate Records Check

In [10]:
duplicates = df.duplicated().sum()
id_duplicates = df["customerID"].duplicated().sum()
print(f"Full duplicate rows: {duplicates}")
print(f"Duplicate customerIDs: {id_duplicates}")

Full duplicate rows: 0
Duplicate customerIDs: 0


## 6. Target Variable Distribution: `Churn`
Understanding target balance is critical. In binary classification, imbalance dictates appropriate evaluation metrics (Accuracy vs Recall / F1 / ROC-AUC).

In [11]:
churn_counts = df["Churn"].value_counts()
churn_percentages = (df["Churn"].value_counts(normalize=True) * 100).round(2)

churn_summary = pd.DataFrame({
    "Count": churn_counts,
    "Percentage (%)": churn_percentages
})
print("Target Variable Distribution:")
print(churn_summary)

Target Variable Distribution:
       Count  Percentage (%)
Churn                       
No      5174           73.46
Yes     1869           26.54


### Class Imbalance Assessment:
- **Retained (No):** 5,174 customers (73.46%)
- **Churned (Yes):** 1,869 customers (26.54%)
- **Imbalance Ratio:** ~2.77 to 1.
- **Methodological Impact:** A naive model predicting "No" for all customers would achieve 73.46% accuracy while identifying **zero** churners. Thus, **Accuracy alone is inadequate**. The model must be optimized and evaluated using **Recall**, **Precision**, **F1-Score**, and **ROC-AUC**.

## 7. Numerical and Categorical Attributes Categorization

In [12]:
numerical_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = [c for c in df.columns if c not in numerical_cols and c not in ["customerID", "Churn"]]

print(f"Numerical Features ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical Features ({len(categorical_cols)}): {categorical_cols}")

Numerical Features (4): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical Features (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [13]:
for col in categorical_cols:
    unique_vals = df[col].unique()
    print(f"{col:20s} ({len(unique_vals):2d} unique) -> {unique_vals}")

gender               ( 2 unique) -> ['Female' 'Male']
Partner              ( 2 unique) -> ['Yes' 'No']
Dependents           ( 2 unique) -> ['No' 'Yes']
PhoneService         ( 2 unique) -> ['No' 'Yes']
MultipleLines        ( 3 unique) -> ['No phone service' 'No' 'Yes']
InternetService      ( 3 unique) -> ['DSL' 'Fiber optic' 'No']
OnlineSecurity       ( 3 unique) -> ['No' 'Yes' 'No internet service']
OnlineBackup         ( 3 unique) -> ['Yes' 'No' 'No internet service']
DeviceProtection     ( 3 unique) -> ['No' 'Yes' 'No internet service']
TechSupport          ( 3 unique) -> ['No' 'Yes' 'No internet service']
StreamingTV          ( 3 unique) -> ['No' 'Yes' 'No internet service']
StreamingMovies      ( 3 unique) -> ['No' 'Yes' 'No internet service']
Contract             ( 3 unique) -> ['Month-to-month' 'One year' 'Two year']
PaperlessBilling     ( 2 unique) -> ['Yes' 'No']
PaymentMethod        ( 4 unique) -> ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (au

## Summary & Key Takeaways
1. **Dimensions:** 7,043 customer records with 21 attributes.
2. **Data Cleanliness:** Zero duplicate records, 11 whitespace strings in `TotalCharges` tied to `tenure=0`.
3. **Target Imbalance:** 26.54% churn rate requires cost-sensitive evaluation (F1 / Recall priority).
4. **Next Step:** Perform Exploratory Data Analysis (EDA) to uncover churn drivers.